# Fine-tune Retriever (BGE-M3) — Modal A100

Chạy giống Kaggle nhưng trên Modal A100 (40 GB). Log stream về notebook.

**Thứ tự:** Cell 2 → Cell 3 → Cell 4 → xong.

In [ ]:
!pip install -q modal

In [ ]:
%%writefile _modal_finetune_retriever.py
import modal, os

image = (
    modal.Image.debian_slim(python_version='3.11')
    .pip_install(
        'torch>=2.4',
        'sentence-transformers>=3.0.0',
        'datasets',
        'huggingface_hub',
        'accelerate>=1.1.0',
        'tqdm',
    )
)

app = modal.App('bge-m3-finetune', image=image)
vol = modal.Volume.from_name('retriever-artifacts', create_if_missing=True)
VOL_PATH = '/vol'


@app.function(gpu='A100', timeout=7200, volumes={VOL_PATH: vol})
def finetune(
    dataset_id: str = 'ariya2357/CORAL',
    dataset_format: str = 'coral',
    baseline_model: str = 'BAAI/bge-m3',
    hf_repo: str = 'trinhtrantran122/bge-m3-coral-retriever',
    hf_token: str = '',
    train_epochs: int = 1,
    batch_size: int = 32,
    learning_rate: float = 2e-5,
    max_doc_char: int = 512,
    max_history_turn_char: int = 128,
    seed: int = 42,
):
    import json, random, re, time
    from pathlib import Path
    import torch
    from datasets import Dataset, load_dataset
    from sentence_transformers import (
        SentenceTransformer, SentenceTransformerTrainer,
        SentenceTransformerTrainingArguments, losses,
    )
    from sentence_transformers.training_args import BatchSamplers
    from tqdm import tqdm

    random.seed(seed)
    Path(VOL_PATH).mkdir(parents=True, exist_ok=True)
    print(f'GPU: {torch.cuda.get_device_name(0)}')

    def normalize_history(raw):
        if not raw: return []
        if isinstance(raw, list): return [str(t).strip() for t in raw if str(t).strip()]
        return [str(raw).strip()]

    def normalize_positive_texts(texts, limit):
        return [str(t).strip()[:limit] for t in (texts or []) if str(t).strip()]

    def build_query(question, history, max_turn_char):
        question = (question or '').strip()
        history = normalize_history(history)
        if len(history) < 2:
            return f'# QUESTION: {question}'
        turns = []
        for i in range(0, len(history) - 1, 2):
            turns.append(f'A: {history[i][:max_turn_char]}\nB: {history[i+1][:max_turn_char]}')
        if not turns:
            return f'# QUESTION: {question}'
        turns.reverse()
        return f'# QUESTION: {question}\n\n[SEP] # HISTORY:\n{"||".join(turns)}'

    def extract_coral_items(ds_dict, splits, limit):
        items = []
        for split in splits:
            print(f'Split {split}: {len(ds_dict[split])} conversations')
            for conv in ds_dict[split]:
                hist = []
                for turn in conv.get('turns', []) or []:
                    q = (turn.get('question') or '').strip()
                    r = (turn.get('response') or '').strip()
                    items.append({'question': q, 'history': list(hist),
                                  'positives': normalize_positive_texts(turn.get('golden_docs_text') or [], limit)})
                    if q: hist.append(q)
                    if r: hist.append(r)
        return items

    def extract_truthreader_items(ds_dict, splits, limit):
        items = []
        for split in splits:
            print(f'Split {split}: {len(ds_dict[split])} rows')
            for item in ds_dict[split]:
                docs = item.get('documents') or []
                positives = [(d.get('document') or d.get('text') or d.get('content') or '') if isinstance(d, dict) else str(d) for d in docs]
                items.append({'question': (item.get('question') or '').strip(),
                              'history': normalize_history(item.get('history')),
                              'positives': normalize_positive_texts(positives, limit)})
        return items

    # --- Build pairs ---
    safe_name = re.sub(r'[^A-Za-z0-9._-]+', '_', dataset_id)
    data_path = os.path.join(VOL_PATH, f'retriever_train_{safe_name}.json')

    if os.path.exists(data_path):
        with open(data_path) as f: pairs = json.load(f)
        print(f'Cached pairs: {len(pairs)}')
    else:
        print(f'Downloading {dataset_id}...')
        ds = (load_dataset(dataset_id, data_files={'train': 'train/new_train_conversation.json'})
              if dataset_format == 'coral' else load_dataset(dataset_id))
        splits = ['train'] if 'train' in ds else list(ds.keys())
        fmt = dataset_format
        if fmt == 'auto':
            fmt = 'coral' if 'turns' in ds[splits[0]][0] else 'truthreader'
        all_items = (extract_coral_items(ds, splits, max_doc_char) if fmt == 'coral'
                     else extract_truthreader_items(ds, splits, max_doc_char))
        all_histories = [it['history'] for it in all_items if len(it['history']) >= 2]
        pairs, skipped = [], 0
        for it in tqdm(all_items, desc='Building pairs'):
            if not it['question'] or not it['positives']:
                skipped += 1; continue
            for pos in it['positives']:
                pairs.append({'query': build_query(it['question'], None, max_history_turn_char), 'positive': pos})
                if len(it['history']) >= 2:
                    pairs.append({'query': build_query(it['question'], it['history'], max_history_turn_char), 'positive': pos})
                    if all_histories:
                        irr = random.choice(all_histories)
                        pairs.append({'query': build_query(it['question'], irr, max_history_turn_char), 'positive': pos})
                        pairs.append({'query': build_query(it['question'], irr + it['history'], max_history_turn_char), 'positive': pos})
        random.shuffle(pairs)
        with open(data_path, 'w') as f: json.dump(pairs, f, ensure_ascii=False)
        vol.commit()
        print(f'Built {len(pairs)} pairs, skipped {skipped}')

    # --- Fine-tune ---
    output_dir = os.path.join(VOL_PATH, 'retriever_model')
    model = SentenceTransformer(baseline_model, device='cuda')
    model.max_seq_length = 512
    train_ds = Dataset.from_dict({'anchor': [p['query'] for p in pairs], 'positive': [p['positive'] for p in pairs]})
    print(f'Training {len(train_ds)} examples, batch={batch_size}')
    total_steps = max(1, len(train_ds) // batch_size)
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir, num_train_epochs=train_epochs,
        per_device_train_batch_size=batch_size, learning_rate=learning_rate,
        warmup_ratio=0.1, bf16=True, logging_steps=max(1, total_steps // 20),
        save_strategy='epoch', batch_sampler=BatchSamplers.NO_DUPLICATES,
        dataloader_num_workers=2, report_to='none',
    )
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_ds,
                                         loss=losses.MultipleNegativesRankingLoss(model))
    t0 = time.time()
    trainer.train()
    model.save(output_dir)
    vol.commit()
    elapsed = round(time.time() - t0, 1)
    print(f'Done in {elapsed}s')

    # --- Push to HF ---
    if hf_repo and hf_token:
        from huggingface_hub import HfApi, create_repo
        with open(os.path.join(output_dir, 'README.md'), 'w') as f:
            f.write(f'# BGE-M3 fine-tuned on CORAL\n- Base: {baseline_model}\n- Epochs: {train_epochs}, Batch: {batch_size}, GPU: A100\n')
        api = HfApi(token=hf_token)
        create_repo(repo_id=hf_repo, token=hf_token, private=True, exist_ok=True)
        api.upload_folder(folder_path=output_dir, repo_id=hf_repo, repo_type='model',
                          commit_message='Upload retriever (Modal A100)')
        print(f'Uploaded: https://huggingface.co/{hf_repo}')
    else:
        print('Skip HF upload.')
    return {'elapsed_sec': elapsed}


@app.local_entrypoint()
def main(
    hf_token: str = '',
    hf_repo: str = 'trinhtrantran122/bge-m3-coral-retriever',
    epochs: int = 1,
    batch_size: int = 32,
):
    print(finetune.remote(hf_token=hf_token, hf_repo=hf_repo, train_epochs=epochs, batch_size=batch_size))

In [ ]:
import subprocess, os

HF_TOKEN = os.getenv('HF_TOKEN', '')  # ← điền token HF vào đây nếu chưa set env

result = subprocess.run(
    ['modal', 'run', '_modal_finetune_retriever.py',
     '--hf-token', HF_TOKEN,
     '--hf-repo', 'trinhtrantran122/bge-m3-coral-retriever',
     '--epochs', '1',
     '--batch-size', '32'],
    text=True
)
print('Exit code:', result.returncode)